# Comparação entre Política Aleatória e Política Básica no Blackjack

## Objetivo
Este notebook compara duas estratégias de decisão para o jogo Blackjack:

- **Política aleatória**, que escolhe ações sem critério;
- **Política básica**, que segue uma regra simples baseada na soma atual da mão do jogador.

O objetivo é criar uma linha de base para avaliar o desempenho de estratégias simples antes de introduzir o agente treinado com Q-Learning.

## Papel desta etapa no projeto

Esta etapa funciona como uma referência inicial para o projeto.

Antes de avaliar um agente treinado com aprendizado por reforço, é importante medir o desempenho de estratégias simples. Assim, os resultados obtidos aqui servirão como base de comparação para as próximas fases.

## Reutilização do ambiente

A lógica do jogo foi centralizada no arquivo `blackjack_env.py` e agora conta com o sistema `ShoeBlackjack` para gerir um baralho finito e contabilizar o True Count.

In [1]:
import pandas as pd
import random
from utils_io import salvar_df

random.seed(42)

In [2]:
from blackjack_env import valor_mao, estado_jogo, ShoeBlackjack

In [3]:
def politica_aleatoria(estado):
    return random.choice(["hit", "stick"])

def politica_basica(estado):
    # O estado agora possui 4 elementos, incluindo o true_count
    soma_jogador, carta_dealer, as_utilizavel, true_count = estado

    if soma_jogador < 17:
        return "hit"
    return "stick"

## Validação inicial da simulação

Testamos a partida adaptando-a para utilizar o novo `ShoeBlackjack`, que rastreia a temperatura do baralho.

In [4]:
def partida_blackjack(policy, shoe, id_episodio=0):
    mao_jogador = [shoe.comprar_carta(), shoe.comprar_carta()]
    mao_dealer = [shoe.comprar_carta(), shoe.comprar_carta()]

    historico = []
    passo = 0

    while True:
        tc = shoe.get_true_count()
        estado = estado_jogo(mao_jogador, mao_dealer[0], tc)
        soma_atual = valor_mao(mao_jogador)

        if soma_atual >= 21:
            break

        acao = policy(estado)

        historico.append({
            "id_episodio": id_episodio,
            "passo": passo,
            "soma_jogador": estado[0],
            "carta_dealer": estado[1],
            "as_utilizavel": estado[2],
            "true_count": estado[3],
            "acao": acao
        })

        if acao == "hit":
            mao_jogador.append(shoe.comprar_carta())

            if valor_mao(mao_jogador) > 21:
                return {
                    "id_episodio": id_episodio,
                    "resultado": "derrota",
                    "recompensa": -1,
                    "total_jogador": valor_mao(mao_jogador),
                    "total_dealer": valor_mao(mao_dealer),
                    "dealer_carta_visivel": mao_dealer[0],
                    "true_count_final": shoe.get_true_count(),
                    "qtd_passos": passo + 1,
                    "mao_jogador": str(mao_jogador),
                    "mao_dealer": str(mao_dealer),
                    "historico": historico
                }
        else:
            break

        passo += 1

    while valor_mao(mao_dealer) < 17:
        mao_dealer.append(shoe.comprar_carta())

    total_jogador = valor_mao(mao_jogador)
    total_dealer = valor_mao(mao_dealer)

    if total_dealer > 21 or total_jogador > total_dealer:
        resultado = "vitoria"
        recompensa = 1
    elif total_jogador < total_dealer:
        resultado = "derrota"
        recompensa = -1
    else:
        resultado = "empate"
        recompensa = 0

    return {
        "id_episodio": id_episodio,
        "resultado": resultado,
        "recompensa": recompensa,
        "total_jogador": total_jogador,
        "total_dealer": total_dealer,
        "dealer_carta_visivel": mao_dealer[0],
        "true_count_final": shoe.get_true_count(),
        "qtd_passos": passo + 1,
        "mao_jogador": str(mao_jogador),
        "mao_dealer": str(mao_dealer),
        "historico": historico
    }

In [5]:
shoe_teste = ShoeBlackjack(num_baralhos=6)
teste = partida_blackjack(politica_basica, shoe=shoe_teste, id_episodio=1)
teste

{'id_episodio': 1,
 'resultado': 'derrota',
 'recompensa': -1,
 'total_jogador': 26,
 'total_dealer': 19,
 'dealer_carta_visivel': 10,
 'true_count_final': 0,
 'qtd_passos': 1,
 'mao_jogador': '[6, 10, 10]',
 'mao_dealer': '[10, 9]',
 'historico': [{'id_episodio': 1,
   'passo': 0,
   'soma_jogador': 16,
   'carta_dealer': 10,
   'as_utilizavel': 0,
   'true_count': 0,
   'acao': 'hit'}]}

## Simulação em múltiplos episódios

Agora o baralho flui naturalmente entre os episódios.

In [6]:
def simular_partidas(policy, nome_politica, n_episodios=100000):
    episodios = []
    decisoes = []
    # Instanciar o shoe fora do loop para manter a contagem contínua
    shoe_continuo = ShoeBlackjack(num_baralhos=6)

    for episodio in range(n_episodios):
        resultado = partida_blackjack(policy, shoe=shoe_continuo, id_episodio=episodio)

        episodios.append({
            "id_episodio": resultado["id_episodio"],
            "politica": nome_politica,
            "resultado": resultado["resultado"],
            "recompensa": resultado["recompensa"],
            "total_jogador": resultado["total_jogador"],
            "total_dealer": resultado["total_dealer"],
            "dealer_carta_visivel": resultado["dealer_carta_visivel"],
            "true_count_final": resultado["true_count_final"],
            "qtd_passos": resultado["qtd_passos"],
            "mao_jogador": resultado["mao_jogador"],
            "mao_dealer": resultado["mao_dealer"]
        })

        for d in resultado["historico"]:
            d["politica"] = nome_politica
            decisoes.append(d)

    df_episodios = pd.DataFrame(episodios)
    df_decisoes = pd.DataFrame(decisoes)

    return df_episodios, df_decisoes

In [7]:
df_epi_aleatoria, df_dec_aleatoria = simular_partidas(
    policy=politica_aleatoria,
    nome_politica="aleatoria",
    n_episodios=100000
)

df_epi_basica, df_dec_basica = simular_partidas(
    policy=politica_basica,
    nome_politica="basica",
    n_episodios=100000
)

In [8]:
df_episodios = pd.concat([df_epi_aleatoria, df_epi_basica], ignore_index=True)
df_decisoes = pd.concat([df_dec_aleatoria, df_dec_basica], ignore_index=True)

df_episodios.head()

,id_episodio,politica,resultado,recompensa,total_jogador,total_dealer,dealer_carta_visivel,true_count_final,qtd_passos,mao_jogador,mao_dealer
0,0,aleatoria,derrota,-1,9,21,10,0,1,"[5, 4]","[10, 1]"
1,1,aleatoria,vitoria,1,20,23,10,-1,1,"[10, 10]","[10, 3, 10]"
2,2,aleatoria,derrota,-1,27,15,5,-1,3,"[1, 3, 9, 4, 10]","[5, 10]"
3,3,aleatoria,vitoria,1,19,18,10,-1,2,"[10, 6, 3]","[10, 8]"
4,4,aleatoria,derrota,-1,26,14,8,0,3,"[7, 4, 4, 1, 10]","[8, 6]"


In [9]:
comparacao = (
    df_episodios
    .groupby(["politica", "resultado"])
    .size()
    .reset_index(name="qtd")
)

comparacao

,politica,resultado,qtd
0,aleatoria,derrota,64003
1,aleatoria,empate,4573
2,aleatoria,vitoria,31424
3,basica,derrota,48733
4,basica,empate,10453
5,basica,vitoria,40814


In [10]:
metricas = (
    df_episodios
    .groupby("politica")
    .agg(
        episodios=("id_episodio", "count"),
        recompensa_media=("recompensa", "mean"),
        total_medio_jogador=("total_jogador", "mean"),
        passos_medios=("qtd_passos", "mean")
    )
    .reset_index()
)

metricas

,politica,episodios,recompensa_media,total_medio_jogador,passos_medios
0,aleatoria,100000,-0.32579,18.30806,1.34464
1,basica,100000,-0.07919,20.31796,1.62601


In [11]:
taxa_vitoria = (
    df_episodios.assign(vitoria=lambda x: (x["resultado"] == "vitoria").astype(int))
    .groupby("politica")["vitoria"]
    .mean()
    .reset_index(name="taxa_vitoria")
)

taxa_vitoria

,politica,taxa_vitoria
0,aleatoria,0.31424
1,basica,0.40814


In [12]:
salvar_df(df_episodios, "blackjack_episodios_etapa_basica", pasta="dados")
salvar_df(df_decisoes, "blackjack_decisoes_etapa_basica", pasta="dados")
salvar_df(metricas, "blackjack_metricas_etapa_basica", pasta="resultados")
salvar_df(taxa_vitoria, "blackjack_taxa_vitoria_etapa_basica", pasta="resultados")

Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_episodios_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\dados\blackjack_decisoes_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_metricas_etapa_basica.csv
Arquivo salvo em: C:\Users\Daianne\Documents\blackjack_RL\resultados\blackjack_taxa_vitoria_etapa_basica.csv
